## PBL Real Project 2 - Wine Quality

1. https://agtechlab.pythonanywhere.com/ 에 접속하여 회원가입해 주세요. (비밀번호는 단순하게 만드는 것을 권장. 예: 1234)
2. `username` 에 이메일 형식의 아이디를 기입해 주세요.
3. `password` 에 비밀번호를 기입해 주세요.

In [ ]:
project = "winequality"  # 수정하지 마세요
username = ""  # 회원가입 시 사용한 이메일아이디 (예시. abc@hello.com)
password = ""  # 비밀번호

리더보드 제출을 위한 기본 설정: 아래 코드를 실행해주세요.


In [2]:
import os
import urllib.request

if not os.path.exists("competition_winequality.py"):
    url = "https://raw.githubusercontent.com/agtechresearch/LectureMLbasic/main/competition/competition_winequality.py"
    filename = "competition_winequality.py"
    urllib.request.urlretrieve(url, filename)

아래 코드를 실행하여 데이터를 다운로드 받습니다: 3개의 csv 파일이 data 폴더에 다운로드됨

 * dataset.csv: 과거 와인 품질평가 데이터 -> 학습에 사용할 데이터셋
 * problem.csv: ML 모델에 의하여 와인 품질을 예측하여야 할 데이터셋
 * submission.csv: 리더보드 서버 제출을 위한 파일 형식


In [ ]:
import competition_winequality as competition

# 파일 다운로드
competition.download_competition_files(project)

Colab 환경에서 한글 폰트 설정을 위한 라이브러리 설치 (한글이 깨져서 나타날 경우 설치 필요)

In [4]:
#!pip install koreanize-matplotlib

----------

## 1. 데이터 불러오기

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 경고 무시
warnings.filterwarnings("ignore")

# Data 경로 설정
DATA_DIR = "datasetwinequality"

In [6]:
# 학습에 사용할 data set 로드 (dataset.csv)
dataset = pd.read_csv(os.path.join(DATA_DIR, "dataset.csv"))

# problem set 로드 (problem.csv)
problemset = pd.read_csv(os.path.join(DATA_DIR, "problem.csv"))

----------

## 2. 데이터셋 확인

▶ 전반적인 데이터셋 정보 확인 (**.info()**) <br>
> 데이터셋의 **컬럼정보와 값의 수, 각 컬럼별 데이터타입, 행의 갯수 등**의 정보 <br>

In [7]:
# 데이터셋 정보 확인
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4798 entries, 0 to 4797
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         4798 non-null   float64
 1   volatile_acidity      4798 non-null   float64
 2   citric_acid           4798 non-null   float64
 3   residual_sugar        4798 non-null   float64
 4   chlorides             4798 non-null   float64
 5   free_sulfur_dioxide   4798 non-null   float64
 6   total_sulfur_dioxide  4798 non-null   float64
 7   density               4798 non-null   float64
 8   pH                    4798 non-null   float64
 9   sulphates             4798 non-null   float64
 10  alcohol               4798 non-null   float64
 11  quality               4798 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 449.9 KB


## [참고] 와인의 특징을 나타내는 항목들 

 - fixed_acidity (고정 산도): 와인에 존재하는 주요 산들 중 타르타르산, 말산 등과 같이 쉽게 증발하지 않는 산의 총량을 나타냅니다. 와인의 풍미와 구조감에 영향을 줍니다.
 
 - volatile_acidity (휘발성 산도): 와인에 존재하는 아세트산(초산)과 같이 쉽게 증발하는 산의 양을 나타냅니다. 너무 높으면 식초 같은 불쾌한 냄새가 날 수 있습니다.

 - citric_acid (시트르산/구연산): 소량으로 발견되며, 와인에 신선함과 풍미를 더할 수 있는 산입니다.
 
 - residual_sugar (잔류 당분): 발효 과정 후 와인에 남아있는 당의 양을 나타냅니다. 와인의 단맛 정도를 결정합니다.

 - chlorides (염화물): 와인에 존재하는 염화물의 양으로, 주로 소금(NaCl)에서 유래합니다. 와인의 맛에 짠맛을 더할 수 있습니다.

 - free_sulfur_dioxide (유리 아황산가스/유리 이산화황): 와인에서 산화 방지 및 미생물 성장 억제제로 사용되는 이산화황 중 다른 화합물과 결합하지 않고 자유로운 형태로 존재하는 부분입니다.

 - total_sulfur_dioxide (총 아황산가스/총 이산화황): 와인에 존재하는 유리 이산화황과 결합된 이산화황의 총량입니다. 와인의 보존성을 높이는 역할을 합니다.

 - density (밀도): 와인의 밀도로, 물의 밀도에 가깝지만 알코올과 당분 함량에 따라 달라집니다. 주로 와인의 바디감을 예측하는 데 사용될 수 있습니다.

 - pH (수소 이온 농도): 와인의 산성도를 나타내는 척도입니다. pH 값이 낮을수록 산도가 높으며, 와인의 색상, 안정성, 맛에 영향을 미칩니다. (보통 3에서 4 사이)

 - sulphates (황산염): 와인에 자연적으로 존재하거나 첨가될 수 있는 황산염 화합물입니다. 항균 작용을 하며 와인의 안정성에 기여할 수 있습니다.

 - alcohol (알코올): 와인에 포함된 에탄올의 함량, 즉 알코올 도수를 나타냅니다. 와인의 바디감, 풍미, 따뜻함에 영향을 줍니다.

 - quality (품질): 와인의 전반적인 품질을 나타내는 등급 (0~10)

----------

## 3. 전처리: 결측치 처리

In [8]:
# 항목별 결측치 비율 확인
dataset.isna().mean()

fixed_acidity           0.0
volatile_acidity        0.0
citric_acid             0.0
residual_sugar          0.0
chlorides               0.0
free_sulfur_dioxide     0.0
total_sulfur_dioxide    0.0
density                 0.0
pH                      0.0
sulphates               0.0
alcohol                 0.0
quality                 0.0
dtype: float64

----------

## 4. 모델 학습 및 평가


In [9]:
# 데이터셋, 독립변수와 종속변수 분리: 독립변수 -> x, 종속변수 -> y
x = dataset.drop(['quality'], axis=1)
y = dataset['quality']

In [10]:
# train set 과 test set 으로 데이터를 나누기 위해 train_test_split 함수를 불러옴
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state=100)

In [11]:
# Random Forest 라이브러리 import
from sklearn.ensemble import RandomForestClassifier

# Random Forest 모델 생성 (예: 100개의 트리 사용)
model = RandomForestClassifier(n_estimators=100, random_state=42) # random_state는 재현성을 위해 설정

# 모델 학습
model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [12]:
from sklearn.metrics import accuracy_score

pred = model.predict(x_test)  # 테스트 데이터로 예측
print("Test accuracy :", accuracy_score(y_test, pred))  # 정확도

Test accuracy : 0.6875


## Problem set 문제에 대한 결혼유무 예측 및 리더보드 결과 제출

- 아래 제출 프로세스가 느리다고 중지 후 다시 코드를 여러차례 재실행하는 경우 패널티가 발생할 수 있습니다. (제출 과정에서 제출 횟수 이슈 발생 가능: 하루 최대 200회 까지 가능)
- 제출에 성공할 경우, "제출에 성공하였습니다"의 메세지와 함께 제출 결과 accuracy 가 화면에 출력됩니다.
- 제출결과는 또한 [대회 페이지(리더보드 서버)](https://agtechlab.pythonanywhere.com/competitions/winequality/)의 `리더보드` 와 `제출` 탭에서 확인할 수 있습니다.


In [13]:
# 문제 데이터(problemset)에 대한 예측값을 구함
problem_pred = model.predict(problemset)

# 리더보드 서버 제출을 위한 파일 생성
submission = pd.read_csv(os.path.join(DATA_DIR, "submission.csv"))
submission["quality"] = problem_pred

# 제출
competition.submit(project, username, password, submission)

아이디:  abc@abc.com
파일명:  submissions\20250529-115113-submission.csv
[제출에 성공하였습니다]
제출 결과: 0.66
